Copyright 2026 Tampere University<br>
This notebook and software was developed for a Tampere University course COMP.CS.320.<br>
This source code is licensed under the MIT license. See LICENSE in the exercise repository root directory.<br>
Author(s): Ville Heikkilä \([ville.heikkila@tuni.fi](mailto:ville.heikkila@tuni.fi))

# COMP.CS.320 Data-Intensive Programming, Exercise 3

This exercise contains additional data processing tasks using DataFrames. The tasks 1-4 concern a new weather dataset, tasks 5-7 a dataset about code commits, and the last task contains theory questions related to lecture content.<br>
The coding tasks can be slightly more challenging than those in exercise 2.

This is the **Python** version, switch to the Scala version if you want to do the tasks in Scala.

Each task has its own cell(s) for the code. Add your solutions to the cells. You are free to add more cells if you feel it is necessary.<br>
There are cells with test code following most of the tasks that involve producing code.<br>
Example outputs for the tasks can be found in a separate notebook: [Exercise 3 example outputs](https://adb-735222077226806.6.azuredatabricks.net/editor/notebooks/8943600422003)

At the end of the notebook, there is a question regarding the use of AI or other collaboration when working the tasks.<br>
Please remember to answer the AI question. And finally, don't forget to submit your solutions to Moodle: [Weekly Exercise #3](https://moodle.tuni.fi/mod/assign/view.php?id=4162926)

##### Some resources that can help with the tasks in this exercise:

- The [tutorial notebook](https://adb-735222077226806.6.azuredatabricks.net/editor/notebooks/3816292345602665) from our course
- Chapters 3 and 5 (Section: Common DataFrames and Spark SQL Operations) in [Learning Spark, 2nd Edition](https://learning.oreilly.com/library/view/learning-spark-2nd/9781492050032/)
    - There are additional code examples in the related [GitHub repository](https://github.com/databricks/LearningSparkV2).
    - The book related notebooks can be imported to Databricks by choosing `import` in your workspace and using the URL<br> `https://github.com/databricks/LearningSparkV2/blob/master/notebooks/LearningSparkv2.dbc`
- [Databricks tutorial](https://docs.databricks.com/en/getting-started/dataframes.html) of using Spark DataFrames
- [Apache Spark documentation](https://spark.apache.org/docs/4.0.4/sql-ref-functions.html) on all available functions that can be used on DataFrames.<br>
  The full [Spark Python functions API listing](https://spark.apache.org/docs/4.0.4/api/python/reference/pyspark.sql/functions.html) for the functions package might have some additional functions listed that have not been updated in the documentation.

In [0]:
# some imports that might be required in the tasks
from pyspark.sql import Column
from pyspark.sql import DataFrame
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Task 1 - Data to DataFrames

The data used in tasks 1-4 is weather data that is based on a dataset from Kaggle: [The Weather Dataset](https://www.kaggle.com/datasets/guillemservera/global-daily-climate-data).<br>

In the [Shared container](https://portal.azure.com/#view/Microsoft_Azure_Storage/ContainerMenuBlade/~/overview/storageAccountId/%2Fsubscriptions%2Fe0c78478-e7f8-429c-a25f-015eae9f54bb%2FresourceGroups%2Ftuni-cs320-f2026-rg%2Fproviders%2FMicrosoft.Storage%2FstorageAccounts%2Ftunics320f2026gen2/path/shared/etag/%220x8DE33DC133306E1%22/defaultId//publicAccessVal/None) in the `exercises/ex3/weather` folder, there are two subfolders: `measurements` and `metadata`.

- `measurements` folder contains daily weather data in Parquet format in file `daily_weather.parquet`.
- `metadata` folder contains two separate CSV files, `cities.csv` and `countries.csv`

Some columns and rows have been removed from the original dataset for this task. For the remaining columns, the Kaggle page has further descriptions on the data.

Parquet is a column-oriented data file format designed for efficient data storage and retrieval. Unlike CSV files, the data given in Parquet format is not as easy to preview without Spark. But if you really want, for example, the [Parquet Visualizer](https://marketplace.visualstudio.com/items?itemName=lucien-martijn.parquet-visualizer) Visual Studio Code extension can be used to browse the data contained in a Parquet file. However, understanding the format is not necessary for this exercise. The format will be discussed in more detail during the lectures later in the course. What is good to know at this point, is that a Parquet file contains metadata about the included columns and their data types, and thus Spark does not need to be given any additional options (unlike with CSV files).

##### The task

- Load the daily weather data into DataFrame `weatherDF`.
- Load the city related metadata into DataFrame `cityDF`.
- Load the country related metadata into DataFrame `countryDF`.

In [0]:
weatherDF: DataFrame = spark.read.parquet("wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex3/weather/measurements/daily_weather.parquet")

cityDF: DataFrame = spark.read.option("header","true").option("sep",",").option("inferSchema","true") \
    .csv("wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex3/weather/metadata/cities.csv")

countryDF: DataFrame = spark.read.option("header","true").option("sep",",").option("inferSchema","true") \
    .csv("wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex3/weather/metadata/countries.csv")

In [0]:
print("The first 5 rows of the weather data")
display(weatherDF.limit(5))

In [0]:
print("The first 5 rows of the city data")
display(cityDF.limit(5))

In [0]:
print("The first 5 rows of the country data")
display(countryDF.limit(5))

## Task 2 - Combined DataFrame

**Part 1:**

- Create a new `metadataDF` DataFrame where the country information from `countryDF` has been added to `cityDF`.
- The new DataFrame should have the same number of rows as the original `cityDF`.
- Hint: either `iso2` or `iso3` column can be used to connect the city and country data.

**Part 2:**

- Create a new `fullDF` DataFrame where the metadata from `metadataDF` has been added to the weather measurements `weatherDF`.
- The new DataFrame should have the same number of rows as the original `weatherDF`.

In the example outputs, some columns that will not be needed in the following tasks have been removed. \
Removing the columns here is allowed, but not compulsory.

In [0]:
metadataDF: DataFrame = cityDF.join(countryDF,on="iso2",how="left_outer").select(cityDF.station_id,cityDF.city_name,countryDF.country,countryDF.capital,countryDF.region)

fullDF: DataFrame = weatherDF.join(metadataDF,on="station_id",how="left_outer").select(weatherDF["*"],metadataDF.city_name, metadataDF.country, metadataDF.capital, metadataDF.region)

In [0]:
print(f"Number of rows in the metadata DataFrame: {metadataDF.count()}")
print("The first 7 rows in the metadata DataFrame:")
metadataDF.limit(7).show(truncate=False)

In [0]:
print(f"Number of rows in the full DataFrame: {fullDF.count()}")
print("The first 7 rows in the full DataFrame:")
fullDF.limit(7).show(truncate=False)

## Task 3 - The coldest and the hottest day in each country

**Part 1:**

- Find the lowest temperature and the date it happened for each country.
- Include the `region` information for each country in the result.
- Order the countries such that the country with the coldest temperature is given first.
- Store the result in `coldDayDF` DataFrame.

**Part 2:**

- Find the highest temperature and the date it happened for each country.
- Again, include the `region` information for each country in the result.
- Order the countries such that the country with the hottest temperature is given first.
- Store the result in `hotDayDF` DataFrame.

In [0]:
query_cold = """
    SELECT 
        w.country, 
        w.region, 
        w.min_temp_c AS coldest_temp, 
        w.date
    FROM {full_df} w
    INNER JOIN (
        SELECT country, MIN(min_temp_c) AS min_temp
        FROM {full_df}
        GROUP BY country
    ) m
    ON w.country = m.country AND w.min_temp_c = m.min_temp
    ORDER BY w.min_temp_c ASC
"""

coldDayDF = spark.sql(query_cold, full_df=fullDF)

query_hot= """
    SELECT 
        w.country, 
        w.region, 
        w.max_temp_c AS hottest_temp, 
        w.date
    FROM {full_df} w
    INNER JOIN (
        SELECT country, MAX(max_temp_c) AS max_temp
        FROM {full_df}
        GROUP BY country
    ) m
    ON w.country = m.country AND w.max_temp_c = m.max_temp
    ORDER BY w.max_temp_c DESC
"""

hotDayDF: DataFrame = spark.sql(query_cold, full_df=fullDF)

In [0]:
coldDayDF.limit(10).show(truncate=False)

In [0]:
hotDayDF.limit(10).show(truncate=False)

## Task 4 - Extreme days in Nordic countries

In this task only the weather data for the independent Nordic countries (`Finland`, `Sweden`, `Norway`, `Denmark`, `Iceland`) should be considered.

Part 1:

- Similarly to task 3, find the coldest and the hottest days (including the dates) for the five Nordic countries.
- Also, find the date with the most rain (including the rain amount) for the five Nordic countries.
- Store the result in `nordicExtremeDayDF` DataFrame with the countries ordered alphabetically.

Part 2:

- Repeat part 1, but only consider data that is measured in the capital city of each Nordic country.
- In this case include the name of the capital city in the results.
- Store the result in `nordicCapitalExtremeDayDF` DataFrame with the countries ordered alphabetically.

In [0]:
# ---------- Part 1 -----------------
nordic_base_df = fullDF.filter(F.col("region") == "Nordic Countries").filter(F.col("country").isin(['Finland', 'Sweden', 'Norway', 'Denmark', 'Iceland']))

min_temp_df = nordic_base_df.groupBy("country").agg(F.min('min_temp_c').alias("coldest_temp"))
coldest_df = nordic_base_df.join(min_temp_df,on="country") \
                            .filter(nordic_base_df.min_temp_c == min_temp_df.coldest_temp) \
                            .select(nordic_base_df.country, F.col("min_temp_c").alias("coldest_temp"), F.col("date").alias("coldest_date"))


max_temp_df = nordic_base_df.groupBy("country").agg(F.max('max_temp_c').alias("hottest_temp"))
hottest_df = nordic_base_df.join(max_temp_df,on="country") \
                            .filter(nordic_base_df.max_temp_c == max_temp_df.hottest_temp) \
                            .select(nordic_base_df.country, F.col("max_temp_c").alias("hottest_temp"), F.col("date").alias("hottest_date"))


max_rain_df = nordic_base_df.groupBy("country").agg(F.max('precipitation_mm').alias("max_rain"))
rainiest_df = nordic_base_df.join(max_rain_df,on="country") \
                            .filter(nordic_base_df.precipitation_mm == max_rain_df.max_rain) \
                            .select(nordic_base_df.country, F.col("precipitation_mm").alias("max_rainfall"), F.col("date").alias("rainiest_day"))


nordicExtremeDayDF: DataFrame = coldest_df \
    .join(hottest_df, on="country", how="inner") \
    .join(rainiest_df, on="country", how="inner") \
    .orderBy("country", ascending=True)

# ---------- Part 2 -----------------

nordic_base_df = fullDF.filter(F.col("region") == "Nordic Countries").filter(F.col("country").isin(['Finland', 'Sweden', 'Norway', 'Denmark', 'Iceland'])).filter(F.col("city_name") == F.col("capital"))

min_temp_df = nordic_base_df.groupBy("country").agg(F.min('min_temp_c').alias("coldest_temp"))
coldest_df = nordic_base_df.join(min_temp_df,on="country") \
                            .filter(nordic_base_df.min_temp_c == min_temp_df.coldest_temp) \
                            .select(nordic_base_df.country, nordic_base_df.capital, F.col("min_temp_c").alias("coldest_temp"), F.col("date").alias("coldest_date"))


max_temp_df = nordic_base_df.groupBy("country").agg(F.max('max_temp_c').alias("hottest_temp"))
hottest_df = nordic_base_df.join(max_temp_df,on="country") \
                            .filter(nordic_base_df.max_temp_c == max_temp_df.hottest_temp) \
                            .select(nordic_base_df.country, F.col("max_temp_c").alias("hottest_temp"), F.col("date").alias("hottest_date"))


max_rain_df = nordic_base_df.groupBy("country").agg(F.max('precipitation_mm').alias("max_rain"))
rainiest_df = nordic_base_df.join(max_rain_df,on="country") \
                            .filter(nordic_base_df.precipitation_mm == max_rain_df.max_rain) \
                            .select(nordic_base_df.country, F.col("precipitation_mm").alias("max_rainfall"), F.col("date").alias("rainiest_day"))

nordicCapitalExtremeDF: DataFrame = coldest_df \
    .join(hottest_df, on="country", how="inner") \
    .join(rainiest_df, on="country", how="inner") \
    .orderBy("country", ascending=True)

In [0]:
print("Extreme days in Nordic countries including measurements from all stations:")
nordicExtremeDayDF.show()

In [0]:
print("Extreme days in Nordic countries using only measurements from country capitals:")
nordicCapitalExtremeDF.show()

## Task 5 - AI assisted commits

In the [Shared container](https://portal.azure.com/#view/Microsoft_Azure_Storage/ContainerMenuBlade/~/overview/storageAccountId/%2Fsubscriptions%2Fe0c78478-e7f8-429c-a25f-015eae9f54bb%2FresourceGroups%2Ftuni-cs320-f2026-rg%2Fproviders%2FMicrosoft.Storage%2FstorageAccounts%2Ftunics320f2026gen2/path/shared/etag/%220x8DE33DC133306E1%22/defaultId//publicAccessVal/None) the `exercises/ex3/commits` folder contains information on the commits (additions/modifications/deletions) made to some code repositories.

The data is from a dataset from Kaggle: [Who Writes AI Code? 1.3M Open-Source AI Commits](https://www.kaggle.com/datasets/abirambharathisa/who-writes-ai-code-1-3m-open-source-ai-commits). The dataset is released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence.
The Kaggle page has descriptions for the data.<br>
In this exercise only the commit data from the first nine months of 2026 is included, and the data was resaved to Parquet format.

Using the commit data, for each month in 2026 find

- the total number of commits that were not merge commits
- the total number of non-merge commits that were not marked as AI assisted
- the total number of non-merge commits that were marked as AI assisted
- the proportion of AI assisted non-merge commits (in percentages, rounded to two decimals)

Give the result as a DataFrame, `monthlyCommitsDF`, with one row per month and the data frame ordered such that January is given first.

In [0]:
commit_df = spark.read.parquet("wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex3/commits")

commit_df = commit_df.filter(F.col("author_datetime_utc") >= "2026-01-01") \
    .filter(F.col("is_merge") == False)
    

monthlyCommitsDF: DataFrame = commit_df.groupBy(F.month("author_datetime_utc").alias("month")).agg(
    F.count("*").alias("commits"),
    F.sum(F.when(F.col("ai_assisted") == False,1)).alias("unassisted_commits"),
    F.sum(F.when(F.col("ai_assisted") == True,1)).alias("assisted_commits"),
    F.round(
        ((F.count("*") - F.sum(F.when(F.col('ai_assisted') == False, 1))) 
         / F.count("*")) * 100, 
        2
    ).alias("assisted_percentage")

    ).orderBy("month").filter(F.col("month")<10)

In [0]:
monthlyCommitsDF.show(truncate=False)

## Task 6 - Repositories and AI assistance

Use the commit data from task 5. Only non merge commits should be considered for the tasks.

**Part 1:**

- Find the repositories that had at least `100` commits but `none` of the commits where marked as AI assisted.
- Store the repository names to `noAIRepos` as a list of strings.

**Part 2**

Considering only repositories which had at least `100` commits and at least `one` AI assisted commit

- Find the four repositories which had the highest AI usage proportionally (assisted commits / total commits).
- Find the four repositories which had the lowest AI usage proportionally (assisted commits / total commits).
- Store the results for both queries in a single DataFrame, `aiReposDF`.
    - In the DataFrame include for each resulting repository
        - the repository name
        - the number of unique authors who have made non-merge commits
        - the total number of non-merge commits
        - the percentage of AI assisted non-merge commits

In [0]:
noAIRepos: list[str] = ???

aiReposDF: DataFrame = ???

In [0]:
print("The repositories with at least 100 commits and no AI-assisted commits (in alphabetical order):")
for repo in noAIRepos:
    print(f"- {repo}")

In [0]:
print("The repositiories with the most and the least AI usage:")
aiReposDF.show(truncate=False)

## Task 7 - AI tools usage

Use the commit data from task 5. Only non merge commits should be considered for the tasks.

- Find the AI tools which have been used for non-merge commits authored in May 2026 at least `100` times.
- For each of these found AI tools calculate the total number of non-merge commits for each month in 2026.
- Collect the results in a single DataFrame, `monthlyToolUsageDF`.
    - See example format for the data frame in the example outputs notebook.

In [0]:
monthlyToolUsageDF: DataFrame = ???

In [0]:
print("The monthly commits with AI tools (considering the most popular AI tools in May:")
monthlyToolUsageDF.show()

## Task 8 - Theory questions

Using your own words, answer the following questions:

- The operations that are used with DataFrames can be divided into transformations and actions.
    1. How would you describe transformations? Give some examples on transformations.
    2. What is the difference between narrow and wide transformations?
    3. How do actions differ from transformation? Give some examples on actions.
- Lazy evaluation has been mentioned in the course.
    4. What does lazy evaluation mean, and why is understanding lazy evaluation important when using Spark?

Extensive answers are not required here.<br>
If your answers do not fit into one screen, you have likely written more than what was expected.

1. ???
2. ???
3. ???
4. ???

## Use of AI and collaboration

Using AI and collaborating with other students is allowed when doing the weekly exercises.<br>
However, the AI use and collaboration should be documented.

- Did you use AI tools while doing this exercise?
  - Did they help? And how did they help?
- Did you work with other students to complete the tasks?
  - Only extensive collaboration is expected to be reported. If you only got help for a couple of the tasks, you don't need to report it here.

???